# RANZ V4 RDM Data Loader

## Quick Start:

1. **Set your load date and table list** (in Cell 2)
2. **Run Cell 2** 
3. **Query your data** using the created temp views

---

## What It Does:

* **Automatically merges** LIVE (`_xref`) and HISTORY (`_hxrf`) tables  
* **Deduplicates** by primary key + EDL_ACT_DTS (keeps latest version)  
* **Creates single temp view** per base table (e.g., `c_b_party`)  
* **Skips duplicate** table processing automatically  
* **Validates data freshness** for LIVE tables only - pipeline FAILS if LIVE partition is stale
* **Optimized code** - consolidated constants, reusable helper functions, no redundancy

---

## Data Freshness Validation:

### Configuration:
* **MAX_DAYS_OLD = 1** (configurable in code)
* **Applies to LIVE tables (_xref) ONLY**

### Behavior:
| Table Type | Freshness Check | Action on Stale Data |
|------------|----------------|----------------------|
| **LIVE (_xref)** | ✅ **Enabled** | **FAILS** pipeline if > 1 day old |
| **HISTORY (_hxrf)** | ❌ **Disabled** | Proceeds with any available partition |

### Error Message Example:
```
ValueError: Data freshness check FAILED: Latest partition is 5 days old (max allowed: 1). 
Expected partition up to 20260519, but latest is 20260514
```

---

## Partition Selection & Filtering Logic:

### LIVE Tables (_xref):

#### Before 2026-04-13:
* **Partition**: Single partition (latest <= Load_Date)
* **Data**: Complete snapshot, no filtering
* ✅ **Freshness**: Must be within 1 day of Load_Date

#### From 2026-04-13 Onwards:
* **Partitions**: ALL partitions from 2026-04-13 to T-1 (Load_Date - 1 day)
* **Data**: Combines complete snapshots from all partitions
* **Deduplication**: By primary key + EDL_ACT_DTS (keeps latest)
* ✅ **Freshness**: Latest partition must be within 1 day of T-1
* **Growth**: Row count increases cumulatively as data accumulates

---

### HISTORY Tables (_hxrf):

* **Partition**: Latest partition <= Load_Date
* ⚠️ **Freshness**: NO CHECK - can use partitions older than MAX_DAYS_OLD
* **Data Filtering by HIST_CREATE_DATE**:

#### One-Time Load (Load_Date ≤ 2026-04-12):
```
HIST_CREATE_DATE: 2025-01-01 to Load_Date
```

#### Ongoing Load (Load_Date ≥ 2026-04-13):
```
Historical Segment: 2025-01-01 to 2026-04-12
Live Segment:       2026-04-13 to T-1
Result:             UNION of both segments
```

---

## Usage Example:

```python
Load_Date = '20260520'

load_df = [
    'c_b_party_xref',
    'c_b_party_hxrf',
    'c_b_contract_xref',
    'c_b_due_diligence_xref',
]

results = load_ranz_v4_rdm_tables(load_df, Load_Date)
```

### Result:

| Table | Partition Selection | Data Coverage | Freshness |
|-------|---------------------|---------------|----------|
| `c_b_party_xref` | 20260413 to 20260519 (36 partitions) | Combined snapshots, deduplicated | ✅ Checked |
| `c_b_party_hxrf` | Latest ≤ 20260520 (1 partition) | Filtered by HIST_CREATE_DATE | ❌ Not checked |
| **Merged View** | - | **UNION + deduplicated** | - |
| `c_b_contract_xref` | 20260413 to 20260519 | Combined snapshots, deduplicated | ✅ Checked |

### Created Temp Views:
* `c_b_party` (merged LIVE + HISTORY)
* `c_b_contract` (LIVE only)
* `c_b_due_diligence` (LIVE only)

---

## Pipeline Behavior:

### ✅ Pipeline SUCCEEDS when:
* **LIVE tables**: Latest partition is current (within MAX_DAYS_OLD threshold)
* **HISTORY tables**: Any partition exists on or before Load_Date
* All required partitions exist for expected date range
* Environment variable `AU_GDP_Defined_Storage_Account` is set

### ❌ Pipeline FAILS when:
* **LIVE tables**: Latest partition too old (> MAX_DAYS_OLD days)
* **LIVE tables**: No partitions in expected date range (2026-04-13 to T-1)
* No partitions found on or before Load_Date
* Environment variable not set
* Both LIVE and HISTORY tables fail for the same base table

---

## Query Your Data:

```sql
-- Query merged view
SELECT * FROM c_b_party LIMIT 10;

-- Check row counts
SELECT COUNT(*) FROM c_b_party;
SELECT COUNT(*) FROM c_b_contract;

-- Analyze data
SELECT 
  SRC_PARTY_ID,
  EDL_ACT_DTS,
  HIST_CREATE_DATE
FROM c_b_party
ORDER BY EDL_ACT_DTS DESC
LIMIT 20;
```

---

## Code Optimizations:

* ✅ Consolidated constants at top (no duplication)
* ✅ Single deduplication function (replaced 3 copies)
* ✅ Reusable helper functions (get_base_table_name, get_primary_key)
* ✅ Extracted common logic (align_schemas, apply_history_date_filter)
* ✅ Cleaner code organization by purpose
* ✅ Reduced function size - single responsibility principle
* ✅ List comprehensions for better performance

In [0]:
# ================================================================================
# RANZ VERSION 4 RDM Data Loader with Auto-Merge - OPTIMIZED
# ================================================================================

from datetime import datetime, timedelta
from typing import Dict, List, Optional
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, row_number, lit
from pyspark.sql.window import Window
import re
import os

# ========== CONSTANTS ==========
CUTOFF_DATE = datetime.strptime('20260412', '%Y%m%d')
ONGOING_START = datetime.strptime('20260413', '%Y%m%d')
HISTORICAL_START = '2025-01-01'

# Data freshness validation (LIVE mode only)
MAX_DAYS_OLD = 1  # Maximum allowed staleness - job will FAIL if LIVE data is older

# Primary key mapping (using BASE table names without suffix)
PRIMARY_KEY_MAP = {
    "ROWID_OBJECT": [
        "c_b_party_rel_addr", "c_b_contract", "c_b_due_diligence",
        "c_b_party_dom_cntry", "c_b_party_naics", "c_b_party_pep_am",
        "c_b_party_rel_party", "c_lkp_cdd_risk_rating", "c_lkp_naics_sector",
        "c_lkp_naics", "c_b_contr_rol_party"
    ],
    "SRC_PARTY_ID": ["c_b_party"]
}

FORCE_LIVE_TABLES = ["c_b_business_line_xref", "c_rbo_rel_type_xref"]

# ========== HELPER FUNCTIONS ==========
def get_base_table_name(table_name: str) -> str:
    """Extract base table name by removing _xref or _hxrf suffix"""
    return table_name.lower().replace('_xref', '').replace('_hxrf', '')

def get_primary_key(base_table_name: str) -> Optional[str]:
    """Get primary key column for a table"""
    for pk_col, tables in PRIMARY_KEY_MAP.items():
        if base_table_name in tables:
            return pk_col
    return None

def deduplicate_dataframe(df: DataFrame, table_name: str) -> DataFrame:
    """Deduplicate DataFrame by primary key + EDL_ACT_DTS"""
    base_name = get_base_table_name(table_name)
    pk_column = get_primary_key(base_name)
    
    if not pk_column or pk_column not in df.columns or 'EDL_ACT_DTS' not in df.columns:
        return df
    
    before_count = df.count()
    window_spec = Window.partitionBy(pk_column).orderBy(col("EDL_ACT_DTS").desc())
    df_deduped = df.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn")
    after_count = df_deduped.count()
    
    if before_count != after_count:
        print(f"Deduplication: {before_count:,} -> {after_count:,} rows (removed {before_count - after_count:,} duplicates)")
    
    return df_deduped

def align_schemas(df1: DataFrame, df2: DataFrame) -> tuple:
    """Align schemas of two DataFrames by adding missing columns"""
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
    
    if cols1 == cols2:
        return df1, df2
    
    all_cols = sorted(cols1.union(cols2))
    
    for col_name in all_cols:
        if col_name not in df1.columns:
            df1 = df1.withColumn(col_name, lit(None))
        if col_name not in df2.columns:
            df2 = df2.withColumn(col_name, lit(None))
    
    return df1.select(all_cols), df2.select(all_cols)

def build_base_path(table_name: str) -> str:
    """Build base path for table storage"""
    storage_account = os.environ.get('AU_GDP_Defined_Storage_Account')
    if not storage_account:
        raise ValueError("Environment variable 'AU_GDP_Defined_Storage_Account' not set")
    return f"abfss://mdm-ranz@{storage_account}.dfs.core.windows.net/{table_name}"

def find_available_partitions(base_path: str) -> List[Dict]:
    """Find all available partitions for a table"""
    partitions = []
    data_path = f"{base_path}/4/data/"
    
    try:
        items = dbutils.fs.ls(data_path)
        for item in items:
            if item.isDir():
                partition_name = item.name.rstrip('/')
                match = re.search(r'LOADED_DTS=(\d{8})T\d{6}Z', partition_name)
                if match:
                    date_str = match.group(1)
                    date_obj = datetime.strptime(date_str, '%Y%m%d')
                    partitions.append({
                        'path': f"{data_path}{partition_name}/",
                        'date': date_obj,
                        'date_str': date_str
                    })
        partitions.sort(key=lambda x: x['date'])
    except Exception as e:
        print(f"Error finding partitions: {e}")
    
    return partitions

# ========== PARTITION SELECTION ==========
def select_live_partitions(partitions: List[Dict], requested_date: datetime) -> List[Dict]:
    """Select all partitions from ongoing_start to T-1 for LIVE mode"""
    t_minus_1 = requested_date - timedelta(days=1)
    selected = [p for p in partitions if ONGOING_START <= p['date'] <= t_minus_1]
    
    if not selected:
        raise ValueError(f"No partitions found between {ONGOING_START.strftime('%Y%m%d')} and {t_minus_1.strftime('%Y%m%d')}")
    
    # FRESHNESS CHECK (LIVE mode only)
    latest_partition = selected[-1]
    days_diff = (t_minus_1 - latest_partition['date']).days
    
    if days_diff > MAX_DAYS_OLD:
        raise ValueError(
            f"Data freshness check FAILED: Latest partition is {days_diff} days old (max allowed: {MAX_DAYS_OLD}). "
            f"Expected partition up to {t_minus_1.strftime('%Y%m%d')}, but latest is {latest_partition['date_str']}"
        )
    
    print(f"LIVE mode: Selected {len(selected)} partitions from {selected[0]['date_str']} to {selected[-1]['date_str']}")
    return selected

def find_closest_partition(partitions: List[Dict], requested_date: datetime, read_mode: str) -> Optional[Dict]:
    """Find closest partition on or before requested date (freshness check only for LIVE mode)"""
    valid_partitions = [p for p in partitions if p['date'] <= requested_date]
    if not valid_partitions:
        raise ValueError(f"No partition found on or before {requested_date.strftime('%Y%m%d')}")
    
    selected_partition = sorted(valid_partitions, key=lambda x: x['date'])[-1]
    days_diff = (requested_date - selected_partition['date']).days
    
    # FRESHNESS CHECK - Only for LIVE mode
    if read_mode == 'LIVE' and days_diff > MAX_DAYS_OLD:
        raise ValueError(
            f"Data freshness check FAILED: Latest partition is {days_diff} days old (max allowed: {MAX_DAYS_OLD}). "
            f"Requested date: {requested_date.strftime('%Y%m%d')}, Latest partition: {selected_partition['date_str']}"
        )
    
    if days_diff > 0:
        if read_mode == 'LIVE':
            print(f"Using partition from {days_diff} day(s) before requested date (within tolerance)")
        else:
            print(f"Using partition from {days_diff} day(s) before requested date")
    
    return selected_partition

# ========== DATA READING ==========
def read_multiple_partitions(partitions: List[Dict], table_name: str) -> DataFrame:
    """Read and combine multiple partitions for LIVE mode"""
    # Read all partitions and union
    dfs = [spark.read.format('delta').load(p['path']) for p in partitions]
    df_combined = dfs[0]
    for df in dfs[1:]:
        df_combined = df_combined.union(df)
    
    total_rows = df_combined.count()
    print(f"\nCombined total: {total_rows:,} rows from {len(partitions)} partitions")
    
    # Deduplicate
    df_deduped = deduplicate_dataframe(df_combined, table_name)
    final_count = df_deduped.count()
    print(f"Final row count: {final_count:,}")
    
    return df_deduped

def apply_history_date_filter(df: DataFrame, requested_datetime: datetime) -> DataFrame:
    """Apply HIST_CREATE_DATE filtering for HISTORY tables"""
    if 'HIST_CREATE_DATE' not in df.columns:
        return df
    
    if requested_datetime <= CUTOFF_DATE:
        # One-time load
        end_date = requested_datetime.strftime('%Y-%m-%d')
        print(f"HISTORY mode (one-time): Filtering HIST_CREATE_DATE from {HISTORICAL_START} to {end_date}")
        return df.filter((col('HIST_CREATE_DATE') >= HISTORICAL_START) & (col('HIST_CREATE_DATE') <= end_date))
    else:
        # Ongoing load: Historical + Live segments
        hist_end = CUTOFF_DATE.strftime('%Y-%m-%d')
        df_historical = df.filter((col('HIST_CREATE_DATE') >= HISTORICAL_START) & (col('HIST_CREATE_DATE') <= hist_end))
        
        t_minus_1 = requested_datetime - timedelta(days=1)
        live_start = ONGOING_START.strftime('%Y-%m-%d')
        live_end = t_minus_1.strftime('%Y-%m-%d') if t_minus_1 >= ONGOING_START else live_start
        
        df_live = df.filter((col('HIST_CREATE_DATE') >= live_start) & (col('HIST_CREATE_DATE') <= live_end))
        print(f"HISTORY mode (ongoing): Historical + Live segments merged")
        return df_historical.union(df_live)

def read_single_partition(partition: Dict, table_name: str, read_mode: str, requested_date: str) -> DataFrame:
    """Read single partition with optional filtering"""
    df = spark.read.format('delta').load(partition['path'])
    initial_count = df.count()
    print(f"Initial row count: {initial_count:,}")
    
    # Apply filtering for HISTORY mode
    if read_mode == 'HISTORY':
        requested_datetime = datetime.strptime(requested_date, '%Y%m%d')
        df = apply_history_date_filter(df, requested_datetime)
    else:
        print(f"LIVE mode: Reading complete snapshot from partition")
    
    # Deduplicate
    df_deduped = deduplicate_dataframe(df, table_name)
    final_count = df_deduped.count()
    print(f"Final row count: {final_count:,}")
    
    return df_deduped

# ========== MAIN READ FUNCTION ==========
def read_ranz_v4_data(table_name: str, requested_date: str) -> DataFrame:
    """Main entry point to read RANZ V4 data"""
    print(f"\n{'='*80}")
    print(f"RANZ-MDM: Processing {table_name} for date {requested_date}")
    print(f"{'='*80}")
    
    # Validate inputs
    if not table_name or not requested_date:
        raise ValueError("Table name and date are mandatory")
    if not re.match(r'^\d{8}$', requested_date):
        raise ValueError(f"Invalid date format: {requested_date}. Expected YYYYMMDD")
    
    requested_datetime = datetime.strptime(requested_date, '%Y%m%d')
    
    # Determine read mode
    table_lower = table_name.lower()
    is_live = table_lower.endswith('_xref') or table_lower in FORCE_LIVE_TABLES
    read_mode = 'LIVE' if is_live else 'HISTORY'
    print(f"Read mode: {read_mode}")
    
    # Find partitions
    base_path = build_base_path(table_name)
    available_partitions = find_available_partitions(base_path)
    
    if not available_partitions:
        raise ValueError(f"No data found for table {table_name}")
    print(f"Available partitions: {len(available_partitions)} found")
    
    # Read data based on mode and date
    if read_mode == 'LIVE' and requested_datetime >= ONGOING_START:
        # Multi-partition read for LIVE tables from cutoff onwards
        selected_partitions = select_live_partitions(available_partitions, requested_datetime)
        return read_multiple_partitions(selected_partitions, table_name)
    else:
        # Single partition read for HISTORY or LIVE before cutoff
        selected_partition = find_closest_partition(available_partitions, requested_datetime, read_mode)
        print(f"Selected partition date: {selected_partition['date_str']}")
        return read_single_partition(selected_partition, table_name, read_mode, requested_date)

# ========== MERGE FUNCTION ==========
def merge_live_and_history(base_table_name: str, requested_date: str) -> DataFrame:
    """Merge LIVE and HISTORY tables"""
    print(f"\n{'='*80}")
    print(f"MERGING: {base_table_name}")
    print(f"{'='*80}")
    
    live_table = f"{base_table_name}_xref"
    history_table = f"{base_table_name}_hxrf"
    
    # Read both tables
    df_live, live_count = None, 0
    df_history, history_count = None, 0
    
    try:
        df_live = read_ranz_v4_data(live_table, requested_date)
        live_count = df_live.count()
    except Exception as e:
        print(f"LIVE table failed: {e}")
    
    try:
        df_history = read_ranz_v4_data(history_table, requested_date)
        history_count = df_history.count()
    except Exception as e:
        print(f"HISTORY table failed: {e}")
    
    # Handle failures
    if df_live is None and df_history is None:
        raise ValueError("Both tables failed")
    if df_live is None:
        return df_history
    if df_history is None:
        return df_live
    
    # Align schemas and union
    df_live, df_history = align_schemas(df_live, df_history)
    df_merged = df_live.union(df_history)
    merged_count = df_merged.count()
    print(f"Merged: {live_count:,} + {history_count:,} = {merged_count:,}")
    
    # Deduplicate across both tables
    pk_column = get_primary_key(base_table_name)
    if pk_column:
        print(f"Deduplicating merged data using primary key: {pk_column}")
        df_merged = deduplicate_dataframe(df_merged, base_table_name)
    else:
        print(f"No deduplication applied - primary key not found")
    
    return df_merged

# ========== MAIN LOAD FUNCTION ==========
def load_ranz_v4_rdm_tables(load_df: list, load_date: str):
    """
    Load RANZ V4 RDM tables with automatic LIVE and HISTORY merging.
    
    Args:
        load_df: List of table names (can include _xref or _hxrf suffixes)
        load_date: Date string in 'YYYYMMDD' format (e.g., '20260520')
    
    Returns:
        List of result dictionaries with status and row counts
    """
    print(f"\n{'='*80}")
    print(f"LOADING RDM DATA OBJECTS - Load Date: {load_date}")
    print(f"{'='*80}")
    
    processed = set()
    results = []
    
    for dataobject in load_df:
        base_name = get_base_table_name(dataobject)
        
        if base_name in processed:
            #print(f"\n[SKIP] '{dataobject}' - already loaded")
            continue
        
        try:
            df_merged = merge_live_and_history(base_name, load_date)
            df_merged.createOrReplaceTempView(base_name)
            row_count = df_merged.count()
            processed.add(base_name)
            results.append({'table': base_name, 'status': 'SUCCESS', 'rows': row_count})
            print(f"\n[SUCCESS] Created view: {base_name} with {row_count:,} rows")
        except Exception as e:
            results.append({'table': base_name, 'status': 'FAILED', 'error': str(e)})
            print(f"\n[FAILED] {base_name} - {e}")    

    
    return results


In [0]:
# ================================================================================
# RANZ VERSION 4 RDM Data Loader with Auto-Merge - OPTIMIZED
# ================================================================================
# ========== CONSTANTS ==========
CUTOFF_DATE = datetime.strptime('20260412', '%Y%m%d')
ONGOING_START = datetime.strptime('20260413', '%Y%m%d')
HISTORICAL_START = '2025-01-01'
 
# Data freshness validation (LIVE mode only)
MAX_DAYS_OLD = 1  # Maximum allowed staleness - job will FAIL if LIVE data is older
 
# Primary key mapping (using BASE table names without suffix)
PRIMARY_KEY_MAP = {
    "ROWID_OBJECT": [
        "c_b_party_rel_addr", "c_b_contract", "c_b_due_diligence",
        "c_b_party_dom_cntry", "c_b_party_naics", "c_b_party_pep_am",
        "c_b_party_rel_party", "c_lkp_cdd_risk_rating", "c_lkp_naics_sector",
        "c_lkp_naics", "c_b_contr_rol_party"
    ],
    "SRC_PARTY_ID": ["c_b_party"]
}
 
FORCE_LIVE_TABLES = ["c_b_business_line_xref", "c_rbo_rel_type_xref"]
 
# ========== HELPER FUNCTIONS ==========
def get_base_table_name(table_name: str) -> str:
    """Extract base table name by removing _xref or _hxrf suffix"""
    return table_name.lower().replace('_xref', '').replace('_hxrf', '')
 
def get_primary_key(base_table_name: str) -> Optional[str]:
    """Get primary key column for a table"""
    for pk_col, tables in PRIMARY_KEY_MAP.items():
        if base_table_name in tables:
            return pk_col
    return None
 
def deduplicate_dataframe(df: DataFrame, table_name: str) -> DataFrame:
    """Deduplicate DataFrame by primary key + EDL_ACT_DTS"""
    base_name = get_base_table_name(table_name)
    pk_column = get_primary_key(base_name)
   
    if not pk_column or pk_column not in df.columns or 'EDL_ACT_DTS' not in df.columns:
        return df
   
    before_count = df.count()
    window_spec = Window.partitionBy(pk_column).orderBy(col("EDL_ACT_DTS").desc())
    df_deduped = df.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn")
    after_count = df_deduped.count()
   
    if before_count != after_count:
        print(f"Deduplication: {before_count:,} -> {after_count:,} rows (removed {before_count - after_count:,} duplicates)")
   
    return df_deduped
 
def align_schemas(df1: DataFrame, df2: DataFrame) -> tuple:
    """Align schemas of two DataFrames by adding missing columns"""
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
   
    if cols1 == cols2:
        return df1, df2
   
    all_cols = sorted(cols1.union(cols2))
   
    for col_name in all_cols:
        if col_name not in df1.columns:
            df1 = df1.withColumn(col_name, lit(None))
        if col_name not in df2.columns:
            df2 = df2.withColumn(col_name, lit(None))
   
    return df1.select(all_cols), df2.select(all_cols)
 
def build_base_path(table_name: str) -> str:
    """Build base path for table storage"""
    storage_account = os.environ.get('AU_GDP_Defined_Storage_Account')
    if not storage_account:
        raise ValueError("Environment variable 'AU_GDP_Defined_Storage_Account' not set")
    return f"abfss://mdm-ranz@{storage_account}.dfs.core.windows.net/{table_name}"
 
def find_available_partitions(base_path: str) -> List[Dict]:
    """Find all available partitions for a table"""
    partitions = []
    data_path = f"{base_path}/4/data/"
   
    try:
        items = dbutils.fs.ls(data_path)
        for item in items:
            if item.isDir():
                partition_name = item.name.rstrip('/')
                match = re.search(r'LOADED_DTS=(\d{8})T\d{6}Z', partition_name)
                if match:
                    date_str = match.group(1)
                    date_obj = datetime.strptime(date_str, '%Y%m%d')
                    partitions.append({
                        'path': f"{data_path}{partition_name}/",
                        'date': date_obj,
                        'date_str': date_str
                    })
        partitions.sort(key=lambda x: x['date'])
    except Exception as e:
        print(f"Error finding partitions: {e}")
   
    return partitions
 
# ========== PARTITION SELECTION ==========
def select_live_partitions(partitions: List[Dict], requested_date: datetime) -> List[Dict]:
    """Select all partitions from ongoing_start to T-1 for LIVE mode"""
    t_minus_1 = requested_date - timedelta(days=1)
    selected = [p for p in partitions if ONGOING_START <= p['date'] <= t_minus_1]
   
    if not selected:
        raise ValueError(f"No partitions found between {ONGOING_START.strftime('%Y%m%d')} and {t_minus_1.strftime('%Y%m%d')}")
   
    # FRESHNESS CHECK (LIVE mode only)
    latest_partition = selected[-1]
    days_diff = (t_minus_1 - latest_partition['date']).days
   
    if days_diff > MAX_DAYS_OLD:
        raise ValueError(
            f"Data freshness check FAILED: Latest partition is {days_diff} days old (max allowed: {MAX_DAYS_OLD}). "
            f"Expected partition up to {t_minus_1.strftime('%Y%m%d')}, but latest is {latest_partition['date_str']}"
        )
   
    print(f"LIVE mode: Selected {len(selected)} partitions from {selected[0]['date_str']} to {selected[-1]['date_str']}")
    return selected
 
def find_closest_partition(partitions: List[Dict], requested_date: datetime, read_mode: str) -> Optional[Dict]:
    """Find closest partition on or before requested date (freshness check only for LIVE mode)"""
    valid_partitions = [p for p in partitions if p['date'] <= requested_date]
    if not valid_partitions:
        raise ValueError(f"No partition found on or before {requested_date.strftime('%Y%m%d')}")
   
    selected_partition = sorted(valid_partitions, key=lambda x: x['date'])[-1]
    days_diff = (requested_date - selected_partition['date']).days
   
    # FRESHNESS CHECK - Only for LIVE mode
    if read_mode == 'LIVE' and days_diff > MAX_DAYS_OLD:
        raise ValueError(
            f"Data freshness check FAILED: Latest partition is {days_diff} days old (max allowed: {MAX_DAYS_OLD}). "
            f"Requested date: {requested_date.strftime('%Y%m%d')}, Latest partition: {selected_partition['date_str']}"
        )
   
    if days_diff > 0:
        if read_mode == 'LIVE':
            print(f"Using partition from {days_diff} day(s) before requested date (within tolerance)")
        else:
            print(f"Using partition from {days_diff} day(s) before requested date")
   
    return selected_partition
 
# ========== DATA READING ==========
def read_multiple_partitions(partitions: List[Dict], table_name: str) -> DataFrame:
    """Read and combine multiple partitions for LIVE mode"""
    # Read all partitions and union
    dfs = [spark.read.format('delta').load(p['path']) for p in partitions]
    df_combined = dfs[0]
    for df in dfs[1:]:
        df_combined = df_combined.union(df)
   
    total_rows = df_combined.count()
    print(f"\nCombined total: {total_rows:,} rows from {len(partitions)} partitions")
   
    # Deduplicate
    df_deduped = deduplicate_dataframe(df_combined, table_name)
    final_count = df_deduped.count()
    print(f"Final row count: {final_count:,}")
   
    return df_deduped
 
def apply_history_date_filter(df: DataFrame, requested_datetime: datetime) -> DataFrame:
    """Apply HIST_CREATE_DATE filtering for HISTORY tables"""
    if 'HIST_CREATE_DATE' not in df.columns:
        return df
   
    if requested_datetime <= CUTOFF_DATE:
        # One-time load
        end_date = requested_datetime.strftime('%Y-%m-%d')
        print(f"HISTORY mode (one-time): Filtering HIST_CREATE_DATE from {HISTORICAL_START} to {end_date}")
        return df.filter((col('HIST_CREATE_DATE') >= HISTORICAL_START) & (col('HIST_CREATE_DATE') <= end_date))
    else:
        # Ongoing load: Historical + Live segments
        hist_end = CUTOFF_DATE.strftime('%Y-%m-%d')
        df_historical = df.filter((col('HIST_CREATE_DATE') >= HISTORICAL_START) & (col('HIST_CREATE_DATE') <= hist_end))
       
        t_minus_1 = requested_datetime - timedelta(days=1)
        live_start = ONGOING_START.strftime('%Y-%m-%d')
        live_end = t_minus_1.strftime('%Y-%m-%d') if t_minus_1 >= ONGOING_START else live_start
       
        df_live = df.filter((col('HIST_CREATE_DATE') >= live_start) & (col('HIST_CREATE_DATE') <= live_end))
        print(f"HISTORY mode (ongoing): Historical + Live segments merged")
        return df_historical.union(df_live)
 
def read_single_partition(partition: Dict, table_name: str, read_mode: str, requested_date: str) -> DataFrame:
    """Read single partition with optional filtering"""
    df = spark.read.format('delta').load(partition['path'])
    initial_count = df.count()
    print(f"Initial row count: {initial_count:,}")
   
    # Apply filtering for HISTORY mode
    if read_mode == 'HISTORY':
        requested_datetime = datetime.strptime(requested_date, '%Y%m%d')
        df = apply_history_date_filter(df, requested_datetime)
    else:
        print(f"LIVE mode: Reading complete snapshot from partition")
   
    # Deduplicate
    df_deduped = deduplicate_dataframe(df, table_name)
    final_count = df_deduped.count()
    print(f"Final row count: {final_count:,}")
   
    return df_deduped
 
# ========== MAIN READ FUNCTION ==========
def read_ranz_v4_data(table_name: str, requested_date: str) -> DataFrame:
    """Main entry point to read RANZ V4 data"""
    print(f"\n{'='*80}")
    print(f"RANZ-MDM: Processing {table_name} for date {requested_date}")
    print(f"{'='*80}")
   
    # Validate inputs
    if not table_name or not requested_date:
        raise ValueError("Table name and date are mandatory")
    if not re.match(r'^\d{8}$', requested_date):
        raise ValueError(f"Invalid date format: {requested_date}. Expected YYYYMMDD")
   
    requested_datetime = datetime.strptime(requested_date, '%Y%m%d')
   
    # Determine read mode
    table_lower = table_name.lower()
    is_live = table_lower.endswith('_xref') or table_lower in FORCE_LIVE_TABLES
    read_mode = 'LIVE' if is_live else 'HISTORY'
    print(f"Read mode: {read_mode}")
   
    # Find partitions
    base_path = build_base_path(table_name)
    available_partitions = find_available_partitions(base_path)
   
    if not available_partitions:
        raise ValueError(f"No data found for table {table_name}")
    print(f"Available partitions: {len(available_partitions)} found")
   
    # Read data based on mode and date
    if read_mode == 'LIVE' and requested_datetime >= ONGOING_START:
        # Multi-partition read for LIVE tables from cutoff onwards
        selected_partitions = select_live_partitions(available_partitions, requested_datetime)
        return read_multiple_partitions(selected_partitions, table_name)
    else:
        # Single partition read for HISTORY or LIVE before cutoff
        selected_partition = find_closest_partition(available_partitions, requested_datetime, read_mode)
        print(f"Selected partition date: {selected_partition['date_str']}")
        return read_single_partition(selected_partition, table_name, read_mode, requested_date)
 
# ========== MERGE FUNCTION ==========
def merge_live_and_history(base_table_name: str, requested_date: str) -> DataFrame:
    """Merge LIVE and HISTORY tables"""
    print(f"\n{'='*80}")
    print(f"MERGING: {base_table_name}")
    print(f"{'='*80}")
   
    live_table = f"{base_table_name}_xref"
    history_table = f"{base_table_name}_hxrf"
   
    # Read both tables
    df_live, live_count = None, 0
    df_history, history_count = None, 0
   
    try:
        df_live = read_ranz_v4_data(live_table, requested_date)
        live_count = df_live.count()
    except Exception as e:
        print(f"LIVE table failed: {e}")
   
    try:
        df_history = read_ranz_v4_data(history_table, requested_date)
        history_count = df_history.count()
    except Exception as e:
        print(f"HISTORY table failed: {e}")
   
    # Handle failures
    if df_live is None and df_history is None:
        raise ValueError("Both tables failed")
    if df_live is None:
        return df_history
    if df_history is None:
        return df_live
   
    # Align schemas and union
    df_live, df_history = align_schemas(df_live, df_history)
    df_merged = df_live.union(df_history)
    merged_count = df_merged.count()
    print(f"Merged: {live_count:,} + {history_count:,} = {merged_count:,}")
   
    # Deduplicate across both tables
    pk_column = get_primary_key(base_table_name)
    if pk_column:
        print(f"Deduplicating merged data using primary key: {pk_column}")
        df_merged = deduplicate_dataframe(df_merged, base_table_name)
    else:
        print(f"No deduplication applied - primary key not found")
   
    return df_merged
 
# ========== MAIN LOAD FUNCTION ==========
def load_ranz_v4_rdm_tables(load_df: list, load_date: str):
    """
    Load RANZ V4 RDM tables with automatic LIVE and HISTORY merging.
   
    Args:
        load_df: List of table names (can include _xref or _hxrf suffixes)
        load_date: Date string in 'YYYYMMDD' format (e.g., '20260520')
   
    Returns:
        List of result dictionaries with status and row counts
    """
    print(f"\n{'='*80}")
    print(f"LOADING RDM DATA OBJECTS - Load Date: {load_date}")
    print(f"{'='*80}")
   
    processed = set()
    results = []
   
    for dataobject in load_df:
        base_name = get_base_table_name(dataobject)
       
        if base_name in processed:
            #print(f"\n[SKIP] '{dataobject}' - already loaded")
            continue
       
        try:
            df_merged = merge_live_and_history(base_name, load_date)
            df_merged.createOrReplaceTempView(base_name)
            row_count = df_merged.count()
            processed.add(base_name)
            results.append({'table': base_name, 'status': 'SUCCESS', 'rows': row_count})
            print(f"\n[SUCCESS] Created view: {base_name} with {row_count:,} rows")
        except Exception as e:
            results.append({'table': base_name, 'status': 'FAILED', 'error': str(e)})
            print(f"\n[FAILED] {base_name} - {e}")    
 
   
    return results